# Kerchunk vs netCDF Read Performance for 3hr Datasets with Slower Kerchunk Reads

Kerchunk read performance for 3hr datasets is slower in certain cases due to dimension order and chunking inefficiencies, particularly for datasets with large `time` dimensions.

**Overview**

- Compares read performance of Kerchunk references and native netCDF files for 3hr datasets.
- Focuses on cases where Kerchunk read times are slower than netCDF due to dimension order and chunking strategies.

**Key Findings**

- Kerchunk datasets with the `time` dimension as the last dimension suffer from inefficiencies due to non-contiguous reads.
- Raw netCDF files, with the `time` dimension as the first dimension, are optimized for sequential access, making them faster for large, contiguous data.
- The absence of chunking in Kerchunk datasets and inconsistent chunks in netCDF datasets exacerbate performance issues, particularly for fragmented datasets.

**Conclusion**

- The performance gap between Kerchunk and netCDF is influenced by the order of dimensions and chunking strategies.
- To improve Kerchunk performance:
  1. Rechunk datasets to make `time` the first dimension.
  2. Align data access patterns with the chunking strategy.
  3. Profile performance to identify and address bottlenecks for specific use cases.


In [1]:
import json

import pandas as pd
import xarray as xr
from IPython.display import HTML

# Prevent truncation of strings
pd.set_option("display.max_colwidth", None)
# Display floats with two decimal places
pd.options.display.float_format = "{:.2f}".format

## Load Raw Metrics


In [2]:
df_raw = pd.read_csv(
    "riotai/results/20260126_130127/kerchunk_vs_netcdf_raw_20260126_130127.csv"
)

# Add a new column for the difference between kerchunk_time and netcdf_time
df_raw["time_difference"] = df_raw["kerchunk_time"] - df_raw["netcdf_time"]

## Check the longest Kerchunk runtimes (Outliers)


In [3]:
df_raw_sorted = df_raw.sort_values(by="time_difference", ascending=False)

df_3hr_slow = (
    df_raw[df_raw["frequency"] == "3hr"]
    .sort_values(by="time_difference", ascending=False)
    .head(n=8)
)

keys = df_3hr_slow["json"].tolist()

HTML(
    f"""
<div style="height:400px; overflow-y:scroll;">
    {df_3hr_slow.to_html(max_rows=None, max_cols=None, notebook=True)}
</div>
"""
)

,frequency,json,num_netcdf_files,timesteps,dims,kerchunk_time,netcdf_time,time_difference
101,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r1i1p2f1.3hr.pr.gr.v20181119.kerchunk.json,100,292200,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 292200}",69.24,49.71,19.53
127,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/control-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.control-1950.r2i1p2f1.3hr.pr.gr.v20190722.kerchunk.json,101,295120,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 295120}",69.84,50.54,19.30
105,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r2i1p2f1.3hr.pr.gr.v20190625.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",45.85,27.76,18.09
100,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-present.r3i1p1f1.3hr.pr.gr.v20190509.kerchunk.json,65,189928,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928}",43.99,27.50,16.49
119,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r1i1p2f1.3hr.pr.gr.v20181212.kerchunk.json,65,189927,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189927}",42.91,30.00,12.91
93,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r3i1p1f1.3hr.pr.gr.v20190713.kerchunk.json,36,105192,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 105192}",23.94,17.20,6.74
104,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/highresSST-future/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-future.r1i1p1f1.3hr.pr.gr.v20190514.kerchunk.json,35,102272,"{'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 102272}",24.31,19.41,4.90
120,3hr,/global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P.highresSST-present.r2i1p1f1.3hr.pr.gr.v20190930.kerchunk.json,65,189928,"{'lat': 256, 'bnds': 2, 'lon': 512, 'time': 189928}",16.16,15.44,0.72


## Analyze the top three slowest file for dimensions and chunking strategy


In [4]:
# Load the JSON file as a dictionary
with open("riotai/json_to_netcdf_maps/json_to_netcdf.json", "r") as file:
    json_netcdf_map = json.load(file)

# Filter the dictionary for entries with the key "3hr"
json_netcdf_3hr_map = json_netcdf_map.get("3hr", {})
json_netcdf_3hr_map = {k: v for k, v in json_netcdf_3hr_map.items() if k in keys}

### Open the datasets (Kerchunk and NetCDF)


In [5]:
ds_dicts = {}

for k, v in list(json_netcdf_3hr_map.items())[:3]:
    ds_kc = xr.open_dataset(k)
    ds_nc = xr.open_mfdataset(v)

    ds_dicts[k] = {"kc": ds_kc, "nc": ds_nc}

/tmp/ipykernel_2310724/1934242529.py:5: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_nc = xr.open_mfdataset(v)
/tmp/ipykernel_2310724/1934242529.py:5: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_nc = xr.open_mfdataset(v)
/tmp/ipykernel_2310724/1934242529.py:5: FutureWarning: In a future version of xarr

### Compare the dimensions


In [8]:
for idx, (k, v) in enumerate(ds_dicts.items()):
    print(f"Dataset: {k}")
    print(f"Kerchunk dimensions: {v['kc'].dims}")
    print(f"NetCDF dimensions : {v['nc'].dims}")
    print("-" * 80)

Dataset: /global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r2i1p2f1.3hr.pr.gr.v20190625.kerchunk.json
Kerchunk dimensions: FrozenMappingWarningOnValuesAccess({'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189928})
NetCDF dimensions : FrozenMappingWarningOnValuesAccess({'time': 189928, 'lat': 512, 'lon': 1024, 'bnds': 2})
--------------------------------------------------------------------------------
Dataset: /global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r1i1p2f1.3hr.pr.gr.v20181212.kerchunk.json
Kerchunk dimensions: FrozenMappingWarningOnValuesAccess({'lat': 512, 'bnds': 2, 'lon': 1024, 'time': 189927})
NetCDF dimensions : FrozenMappingWarningOnValuesAccess({'time': 189927, 'lat': 512, 'lon': 1024, 'bnds': 2})
--------------------------------------------------------------------------------
Dataset: /global/cfs/projectdirs/m

**Observations:**

- The Kerchunk datasets have the `time` dimension as the last dimension, while the netCDF datasets opened with `open_mfdataset` has the `time` dimension as the first dimension.
- Data is stored in row-major order (C-order), meaning the last dimension is stored contiguously in memory.

**Why Dimension Order Matters:**

- When the `time` dimension is last and large, accessing it requires reading many non-contiguous chunks, leading to inefficiencies in Kerchunk.
- Raw netCDF files are optimized for sequential access, making them faster when reading large, contiguous data along the last dimension.

**Recommendations:**

1. **Rechunk the Dataset**: Reorder the dimensions so that `time` is the first dimension for better Kerchunk performance.
2. **Optimize Access Patterns**: Align data access with the chunking strategy to minimize non-contiguous reads.
3. **Profile Performance**: Compare Kerchunk and raw netCDF for specific use cases to identify bottlenecks and optimize workflows.


### Check the chunking strategy


In [10]:
for idx, (k, v) in enumerate(ds_dicts.items()):
    print(f"Dataset: {k}")
    print(f"Kerchunk chunks: {v['kc'].chunks}")

    try:
        print(f"NetCDF chunks : {v['nc'].chunks}")
    except ValueError as e:
        print(f"NetCDF chunks : {e}")

    print("-" * 80)

Dataset: /global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r2i1p2f1.3hr.pr.gr.v20190625.kerchunk.json
Kerchunk chunks: Frozen({})
NetCDF chunks : Object has inconsistent chunks along dimension time. This can be fixed by calling unify_chunks().
--------------------------------------------------------------------------------
Dataset: /global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/hist-1950/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.hist-1950.r1i1p2f1.3hr.pr.gr.v20181212.kerchunk.json
Kerchunk chunks: Frozen({})
NetCDF chunks : Object has inconsistent chunks along dimension time. This can be fixed by calling unify_chunks().
--------------------------------------------------------------------------------
Dataset: /global/cfs/projectdirs/m4931/sasha-tmp/kerchunk/pr/highresSST-present/hr-misc/CMIP6.HighResMIP.EC-Earth-Consortium.EC-Earth3P-HR.highresSST-present.r3i1p1f1.3hr.pr.gr.v20190509.k

**Observation:**

- Kerchunk dataset has no chunking strategy applied.
- netCDF dataset has inconsistent chunks along dimension time.

**Interpretation:**

- The lack of chunking in the Kerchunk dataset means that data access is not optimized for specific dimensions, leading to inefficiencies when accessing large datasets with high temporal resolution.
- Inconsistent chunks in the netCDF dataset along the `time` dimension can cause performance degradation, as `open_mfdataset` needs to reconcile these inconsistencies, adding overhead.
- The combination of large `time` dimensions and the absence of optimized chunking in Kerchunk exacerbates the performance gap, particularly for datasets with high temporal resolution and fragmentation.


## Overall Takeaway

### Key factors
- Performance differences between Kerchunk and raw netCDF are mainly driven by:
  - **Dimension order**
  - **Chunking strategy**

### Dimension order
- Kerchunk datasets with `time` as the **last dimension** are inefficient:
  - Time slicing requires **non-contiguous reads**
  - Costs grow with large domains and high temporal resolution
- Raw netCDF files usually place `time` **first**:
  - Optimized for **contiguous, sequential reads**
  - Faster for common access patterns

### Why Kerchunk makes `time` the last dimension
- Kerchunk constructs a **virtual dataset** from references, not from a single on-disk layout
- It prioritizes:
  - Reusing existing chunk layouts
  - Minimizing metadata and rewrite costs
- Many source netCDF files are chunked with `time` **not** as the leading chunked dimension
- When aggregating many small time-slice files:
  - Kerchunk concatenates along `time`
  - The resulting virtual layout often places `time` **last**
- This reflects **how the data are chunked**, not how they were originally written

### Chunking
- Lack of chunking in Kerchunk datasets increases I/O overhead
- Inconsistent chunking in netCDF datasets also degrades performance

### Improving Kerchunk performance
- Rechunk so `time` is the **first dimension**
- Match access patterns to chunk layout
- Profile workloads to target bottlenecks
